# Universe Coverage (Slices 6A–6C)

OwnerLens's SEC-first canonical pipeline was validated on a handful of companies. Before designing the next normalization improvement, this notebook measures how it holds up across **24 conventional U.S. operating companies** in several industries:

```
SEC fetch → canonical history → owner economics → capital efficiency → economic value
          → compounding → capital allocation → summary → coverage
```

This is **coverage discovery**. Nothing here changes SEC normalization or adds ticker overrides. Every blocker is diagnosed from the raw Company Facts using a small read-only catalog of alternative SEC concepts.

The survey was run with `uv run python scripts/survey_universe.py`, which ingests through the existing `ingest_company` into a dedicated `data/universe_6a.db`. This notebook replays it **offline** from the stored raw snapshots, so it is reproducible and makes no SEC requests.

In [1]:
import json
from pathlib import Path

from owner_lens.persistence import FilesystemRawSnapshotStore
from owner_lens.universe import (
    UNIVERSE_6A,
    UniverseReport,
    format_company_view,
    format_metric_view,
    format_pattern_view,
    survey_from_snapshot,
)
from owner_lens.universe import DiagnosisCategory as D
from owner_lens.universe import RestatementKind as R

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
manifest = json.loads((repo_root / 'data' / 'universe_6a_report.json').read_text(encoding='utf-8'))
raw_store = FilesystemRawSnapshotStore(repo_root / 'data' / 'raw' / 'sec' / 'company_facts')
report = UniverseReport(tuple(survey_from_snapshot(entry, raw_store) for entry in manifest['companies']))
print(f'{len(report.companies)} companies replayed offline from stored SEC snapshots')
for member in UNIVERSE_6A:
    if member.note:
        print(f'  {member.ticker}: {member.note}')

24 companies replayed offline from stored SEC snapshots
  CAT: stress test: captive finance arm (Cat Financial)
  DE: stress test: captive finance arm (John Deere Financial)
  UNH: stress test: insurance-like balance sheet


## 1. Company view

Each row shows the six analytical layers (`OK` available, `PART` partial, `INSUF` insufficient data, `UNAV` unsupported input, `BLOCK` invalid input, `ERR` unexpected exception) and the first blocking metric with its diagnosis.

`FULL` requires every layer to be available **and** no silently stale metric.

In [2]:
print(format_company_view(report))

Ticker Overall  Owner CapEf EcVal Comp  CapAl Summ   Unsupported                        Invalid                  Stale (silent)               Primary blocker
ADBE   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
V      PARTIAL  PART  OK    INSUF INSUF PART  INSUF  diluted_shares                     -                        -                            diluted_shares [MISSING_CONCEPT]
COST   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
MSFT   FULL     OK    OK    OK    OK    OK    OK     -                                  -                        -                            -
CRM    PARTIAL  OK    UNAV  UNAV  UNAV  UNAV  UNAV   current_debt                       long_term_debt           -                            current_debt [ALTERNATIVE_CONCEPT]
NOW    PARTIAL  OK    UNAV  UNAV  UNAV  UNAV  UNAV   curre

## 2. Metric view: which canonical metrics block coverage?

For each canonical metric: how many companies have it available, unsupported, invalid (ambiguous or malformed), or structurally absent; how many companies it blocks; and the dominant diagnosis.

In [3]:
print(format_metric_view(report))

Metric                    Avail Unsup Inval Absent Blocks  Dominant diagnosis   Companies affected
revenue                      22     1     1      0      2  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: PFE
operating_income             19     5     0      0      5  MISSING_CONCEPT      unsupported: NKE,DE,PFE,XOM,CVX
net_income                   22     2     0      0      1  STALE_CONCEPT        unsupported: CAT,XOM
operating_cash_flow          23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
capital_expenditures         22     1     1      0      1  AMBIGUOUS_DUPLICATE  unsupported: XOM; invalid: META
diluted_shares               22     2     0      0      1  MISSING_CONCEPT      unsupported: V,XOM
income_tax_expense           23     1     0      0      0  PERIOD_ISSUE         unsupported: XOM
pretax_income                22     2     0      0      1  STALE_CONCEPT        unsupported: ORCL,XOM
repurchases                  21     3     0      0      0  MISSING_CON

## 3. Patterns: recurring taxonomy versus company quirks

In [4]:
print(format_pattern_view(report))

Top blocking metrics (companies blocked):
  current_debt              9
  operating_income          5
  cash                      2
  revenue                   2
  capital_expenditures      1
Diagnosis categories (company-metric findings):
  ALTERNATIVE_CONCEPT    9
  AMBIGUOUS_DUPLICATE    5
  BUSINESS_STRUCTURE     1
  COMPOSITE_CANDIDATE    5
  MISSING_CONCEPT        16
  PERIOD_ISSUE           13
  STALE_CONCEPT          14
Silently stale AVAILABLE metrics (selected concept has no recent value): 0 in 0 companies
Ambiguous duplicates by restatement kind:
  VALUE_CHANGE    5  CRM:long_term_debt, META:capital_expenditures, INTU:long_term_debt, PFE:revenue, PFE:cash
Conflicts resolved by the normalizer (precision / stock split): 15
  NOW    net_income             FY2021 PRECISION
  NOW    diluted_shares         FY2024 STOCK_SPLIT
  NOW    diluted_shares         FY2023 STOCK_SPLIT
  NOW    diluted_shares         FY2022 STOCK_SPLIT x5
  NOW    diluted_shares         FY2021 STOCK_SPLIT x5

## 4. The most important finding: silent staleness (fixed in Slice 6B)

In the 6A survey the SEC selector took the first preferred concept with *any* annual history. When a company stopped tagging that concept, OwnerLens kept the old values and marked the metric `AVAILABLE`. 14 metrics in 10 companies were affected, for example NVIDIA revenue (ending FY2022), Amazon capital expenditures (ending FY2016), and Caterpillar net income (ending FY2010).

Slice 6B made concept selection recency-aware. A concept is eligible only if it covers the company's latest fiscal year, so stale history can no longer appear as `AVAILABLE`. A current fallback concept is selected when one exists in the preference list; otherwise the metric is `UNSUPPORTED` and diagnosed `STALE_CONCEPT`. The cell below lists any silently stale metric. After 6B it should print nothing.

In [5]:
for company in report.surveyed:
    for metric in company.silently_stale():
        print(f'{company.ticker:<5} {metric:<24} {company.metrics[metric].reason[:120]}')

## 4b. Restatement and stock-split conflicts (resolved in Slice 6C)

The same fiscal year often has several reported values because later 10-Ks restate comparatives. 6C resolves only the mechanical cases. **Precision re-roundings** are an exact rounding to thousands through millions, never a percentage band. **Stock splits** apply to share counts only, need an integer forward ratio and a consistent filing order, and older pre-split-only years are scaled to the post-split basis. Genuine revisions still fail loudly. Every resolved value keeps its as-filed value, the split factor, and the superseded reports.

In [6]:
print(f'Resolved conflicts: {len(report.resolved_conflicts())}')
for ticker, metric, entry in report.resolved_conflicts():
    print(f'  {ticker:<5} {metric:<22} {entry}')
print()
print('Still failing loudly (genuine revisions):')
for kind, items in report.restatements().items():
    print(f'  {kind:<13} {", ".join(items)}')

Resolved conflicts: 15


  NOW   net_income             FY2021 PRECISION
  NOW   diluted_shares         FY2024 STOCK_SPLIT
  NOW   diluted_shares         FY2023 STOCK_SPLIT
  NOW   diluted_shares         FY2022 STOCK_SPLIT x5
  NOW   diluted_shares         FY2021 STOCK_SPLIT x5
  NOW   cash                   FY2020 PRECISION
  NOW   total_assets           FY2020 PRECISION
  NOW   total_equity           FY2020 PRECISION
  AMZN  diluted_shares         FY2021 STOCK_SPLIT
  NVDA  diluted_shares         FY2024 STOCK_SPLIT
  NVDA  diluted_shares         FY2023 STOCK_SPLIT
  NVDA  diluted_shares         FY2022 STOCK_SPLIT x10
  CMG   diluted_shares         FY2023 STOCK_SPLIT
  CMG   diluted_shares         FY2022 STOCK_SPLIT
  CMG   diluted_shares         FY2021 STOCK_SPLIT x50

Still failing loudly (genuine revisions):


  VALUE_CHANGE  CRM:long_term_debt, META:capital_expenditures, INTU:long_term_debt, PFE:revenue, PFE:cash


In [7]:
from owner_lens.sec_adapter import canonical_history_from_sec

nvda = next(entry for entry in manifest['companies'] if entry['ticker'] == 'NVDA')
history = canonical_history_from_sec(json.loads(raw_store.get(nvda['content_hash'])), ticker='NVDA')
for fact in history.series_for('diluted_shares').observations:
    r = fact.resolution
    detail = '' if r is None else (
        f'  {r.kind.value} x{r.split_factor}, as filed {r.reported_value:,}; superseded '
        + ', '.join(f'{s.value:,} ({s.filed})' for s in r.superseded)
    )
    print(f'FY{fact.fiscal_year} {fact.value:>16,}  {fact.accession}{detail}')

FY2026   24,514,000,000  0001045810-26-000021
FY2025   24,804,000,000  0001045810-25-000023
FY2024   24,940,000,000  0001045810-25-000023  STOCK_SPLIT x1, as filed 24,940,000,000; superseded 2,494,000,000 (2024-02-21)
FY2023   25,070,000,000  0001045810-25-000023  STOCK_SPLIT x1, as filed 25,070,000,000; superseded 2,507,000,000 (2023-02-24), 2,507,000,000 (2024-02-21)
FY2022   25,350,000,000  0001045810-22-000036  STOCK_SPLIT x10, as filed 2,535,000,000; superseded 


## 5. What would the next normalization improvement unblock?

`unblocked_by` is a what-if over the diagnoses. It counts companies whose every unsupported or invalid required input would be resolved by an improvement. It does not recompute anything, and it ignores silent staleness, so it measures blocked coverage, not trustworthiness.

In [8]:
current_debt = lambda t, f: f.metric == 'current_debt' and f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.COMPOSITE_CANDIDATE, D.MISSING_CONCEPT)
ambiguity = lambda t, f: f.diagnosis is D.AMBIGUOUS_DUPLICATE and f.restatement in (R.PRECISION, R.STOCK_SPLIT)
fallback = lambda t, f: f.diagnosis in (D.ALTERNATIVE_CONCEPT, D.STALE_CONCEPT) and bool(f.candidates)
long_term_debt = lambda t, f: f.metric == 'long_term_debt' and f.diagnosis is not D.PERIOD_ISSUE

scenarios = {
    'A. current-debt policy (fallbacks + composite + absence)': current_debt,
    'B. restatement-aware ambiguity (delivered in 6C)': ambiguity,
    'C. catalog concept policy': fallback,
    'A + B': lambda t, f: current_debt(t, f) or ambiguity(t, f),
    'A + B + C': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f),
    'A + B + C + long-term debt': lambda t, f: current_debt(t, f) or ambiguity(t, f) or fallback(t, f) or long_term_debt(t, f),
}
blocked = [c.ticker for c in report.surveyed if c.unusable_inputs()]
print(f'{len(blocked)} of {len(report.companies)} companies have at least one blocking input\n')
for name, resolves in scenarios.items():
    unblocked = report.unblocked_by(resolves)
    print(f'{name:<60} unblocks {len(unblocked):>2}: {", ".join(unblocked)}')

19 of 24 companies have at least one blocking input

A. current-debt policy (fallbacks + composite + absence)     unblocks  3: CMG, HD, LOW
B. restatement-aware ambiguity (delivered in 6C)             unblocks  0: 
C. catalog concept policy                                    unblocks  5: PG, HD, LOW, CAT, UNH
A + B                                                        unblocks  3: CMG, HD, LOW
A + B + C                                                    unblocks  7: CMG, PG, KO, HD, LOW, CAT, UNH
A + B + C + long-term debt                                   unblocks 10: CRM, INTU, LULU, CMG, PG, KO, HD, LOW, CAT, UNH


## 6. Acceptance answers

In [9]:
total = len(report.companies)
dist = report.distribution()
print('1-2. Coverage:', ', '.join(f'{k} {v}/{total} ({v / total:.0%})' for k, v in dist.items()))
print('3.   Top blocking metrics:', ', '.join(f'{m} ({n})' for m, n in report.top_blockers()))
recurring = [r for r in report.recurring_candidates() if r[3]]
print(f'4.   Recurring alternative concepts (3+ companies): {len(recurring)};',
      'quirks:', len(report.recurring_candidates()) - len(recurring))
composites = report.composites()
print(f'5.   Composite candidates: {len(composites)} in {len({c[0] for c in composites})} companies',
      f'({sum(1 for c in composites if c[2])} value-level understatements)')
stale = [c for c in report.surveyed if c.silently_stale()]
print(f'     Silently stale metrics: {sum(len(c.silently_stale()) for c in stale)} in {len(stale)} companies')
print('     Diagnosis mix:', report.diagnosis_counts())

1-2. Coverage: FULL 4/24 (17%), PARTIAL 13/24 (54%), FAILED 7/24 (29%)


3.   Top blocking metrics:

 current_debt (9), operating_income (5), cash (2), revenue (2), capital_expenditures (1)
4.   Recurring alternative concepts (3+ companies): 6; quirks: 6
5.   Composite candidates: 5 in 5 companies (2 value-level understatements)
     Silently stale metrics: 0 in 0 companies
     Diagnosis mix: {'ALTERNATIVE_CONCEPT': 9, 'AMBIGUOUS_DUPLICATE': 5, 'BUSINESS_STRUCTURE': 1, 'COMPOSITE_CANDIDATE': 5, 'MISSING_CONCEPT': 16, 'PERIOD_ISSUE': 13, 'STALE_CONCEPT': 14}


**Reading the answers**

The prose below records the original **6A** survey. The cells above replay the stored snapshots with the *current* code, so they show the results after Slice 6B (recency-aware selection) and Slice 6C (conflict resolution). The before and after comparisons are in the Feature 6 documentation.

* **Coverage:** in 6A, 3 of 24 companies (12%) reached full coverage, 12 (50%) were partial, and 9 (38%) failed. A failure means owner economics itself is unavailable. After 6B: 3 full, 10 partial, 11 failed; CAT and DE now fail honestly on stale or discontinued concepts. After 6C: **4 full (adds NVDA), 13 partial, 7 failed**. NOW, AMZN, and CMG gained owner economics.
* **Top blockers:** after 6C, current debt dominates (9 companies), followed by operating income (5: energy, pharma, Nike, Deere). Diluted shares no longer block anyone through split restatements.
* **Recurring versus quirk:** failures are **mostly recurring taxonomy patterns**. Three current-debt alternatives each recur in 3–4 companies. Of the 6A conflicts, the exact 6C rule confirms 4 re-roundings and 4 stock splits. The other 5 are genuine revisions (CRM, INTU, META, PFE), which 6A's 1% heuristic had partly mislabelled as precision. Staleness recurred across 10 companies.
* **Composite metrics:** current debt is composite in 5 companies (3 unsupported, 2 value-level understatements, including MSFT FY2024 commercial paper). No other metric needed a composite.
* **SEC-first at 100+:** practical, *if* improvements stay general rules rather than per-ticker overrides. 6B and 6C each moved coverage with no override.
* **Next improvement:** a current-debt policy, then a catalog-concept policy. Genuine revisions, operating-income derivation, and discontinued activity lines remain separate decisions.